In [1]:
import spacy
from spacy.training import Example
from spacy.util import minibatch, compounding
import random
import json
from spacy.util import filter_spans

In [2]:
with open('Data/train/train_data.json','rb') as f:
    train_data=json.load(f)

In [3]:
# create a blank English NLP model
nlp = spacy.blank('en')

# Create the NER component and add it to the pipeline
if "ner" not in nlp.pipe_names:
    ner = nlp.add_pipe("ner", last=True)
else:
    ner = nlp.get_pipe("ner")

# Add labels to the NER component
for item in train_data:
    for _, _, label in item['entities']:
        ner.add_label(label)

# Prepare training data in the format required by spaCy 3.x
train_examples = []
count=0
for item in train_data:
    doc = nlp.make_doc(item["text"])
    ents = []
    for start, end, label in item['entities']:
        span = doc.char_span(start, end, label=label, alignment_mode="contract")
        if span is not None:
            ents.append(span)
    
    filtered_ents = filter_spans(ents)
    doc.ents = filtered_ents
    example = Example.from_dict(doc, {"entities": item['entities']})
    train_examples.append(example)



C:\Users\Sachin\AppData\Roaming\Python\Python313\site-packages\spacy\training\iob_utils.py:149: UserWarning: [W030] Some entities could not be aligned in the text "Ananya Chavan lecturer - oracle tutorials  Mumbai,..." with entities "[[857, 860, 'DEGREE'], [973, 1703, 'SKILLS'], [43,...". Use `spacy.training.offsets_to_biluo_tags(nlp.make_doc(text), entities)` to check the alignment. Misaligned entities ('-') will be ignored during training.
  warnings.warn(
C:\Users\Sachin\AppData\Roaming\Python\Python313\site-packages\spacy\training\iob_utils.py:149: UserWarning: [W030] Some entities could not be aligned in the text "Imgeeyaul Ansari java developer  Pune, Maharashtra..." with entities "[[1894, 2173, 'SKILLS'], [1726, 1850, 'SKILLS'], [...". Use `spacy.training.offsets_to_biluo_tags(nlp.make_doc(text), entities)` to check the alignment. Misaligned entities ('-') will be ignored during training.
  warnings.warn(
C:\Users\Sachin\AppData\Roaming\Python\Python313\site-packages\spacy\train

In [4]:
# Initialize the optimizer
optimizer = nlp.begin_training()

# Training loop
n_iter = 300
for itn in range(n_iter):
    random.shuffle(train_examples)
    losses = {}
    # Batch up the examples using spaCy's minibatch
    batches = minibatch(train_examples, size=compounding(4.0, 32.0, 1.001))
    for batch in batches:
        nlp.update(
            batch,  # batch of Example objects
            drop=0.2,  # dropout - make it harder to memorise data
            sgd=optimizer,  # callable to update weights
            losses=losses
        )
    scores = nlp.evaluate(train_examples)
    ents_p = scores["ents_p"]
    ents_r = scores["ents_r"]
    ents_f = scores["ents_f"]

    print(f"Iteration {itn+1}: Losses: {losses['ner']:.3f}, Precision: {ents_p:.3f}, Recall: {ents_r:.3f}, F1-score: {ents_f:.3f}")

# Save the model
nlp.to_disk("ner_model")

Iteration 1: Losses: 31785.625, Precision: 0.991, Recall: 1.000, F1-score: 0.996
Iteration 2: Losses: 5778.447, Precision: 0.933, Recall: 1.000, F1-score: 0.966
Iteration 3: Losses: 4147.606, Precision: 0.953, Recall: 1.000, F1-score: 0.976
Iteration 4: Losses: 3188.049, Precision: 0.952, Recall: 1.000, F1-score: 0.976
Iteration 5: Losses: 2863.039, Precision: 0.911, Recall: 1.000, F1-score: 0.953
Iteration 6: Losses: 2444.118, Precision: 0.864, Recall: 1.000, F1-score: 0.927
Iteration 7: Losses: 2349.613, Precision: 0.931, Recall: 1.000, F1-score: 0.964
Iteration 8: Losses: 2139.146, Precision: 0.908, Recall: 1.000, F1-score: 0.952
Iteration 9: Losses: 1970.644, Precision: 0.834, Recall: 1.000, F1-score: 0.910
Iteration 10: Losses: 1875.856, Precision: 0.893, Recall: 1.000, F1-score: 0.944
Iteration 11: Losses: 1780.304, Precision: 0.907, Recall: 1.000, F1-score: 0.951
Iteration 12: Losses: 1630.856, Precision: 0.904, Recall: 1.000, F1-score: 0.949
Iteration 13: Losses: 1557.553, Prec